In [6]:
import pandas as pd
import numpy as np
import glob
import os

def calculate_average_snr(folder_path, file_pattern="*.csv", noise_floor_dbm=-87.98):
    """
    指定されたフォルダ内のCSVファイルを読み込み、条件に従ってSNRを計算し、その平均を返します。
    
    条件:
    1. データ部分の最初の3行以外を使用
    2. 最大値を検出してdBmに変換
    3. ノイズフロア(-81dBm)との差をSNRとする
    """
    
    # 検索パスの作成
    search_path = os.path.join(folder_path, file_pattern)
    files = sorted(glob.glob(search_path))
    
    if not files:
        print(f"エラー: 指定されたフォルダ '{folder_path}' にファイルが見つかりません。")
        return

    snr_values = []
    print(f"処理対象ファイル数: {len(files)}")

    for file in files:
        try:
            # CSVファイルの読み込み
            # ヘッダーがある場合はそのまま読み込みます。ヘッダーがない場合は header=None を指定してください。
            df = pd.read_csv(file)
            
            # 電力データが含まれる列を取得（2列目と仮定: インデックス1）
            # ※実際のファイル形式に合わせて iloc[:, 1] の数値を変更してください
            if df.shape[1] < 2:
                print(f"警告: ファイル {os.path.basename(file)} の列数が不足しています。スキップします。")
                continue
                
            data_series = df.iloc[:, 1]
            
            # データ部分の最初の3行を除外（インデックス3以降を使用）
            # df.iloc[3:] は 4行目以降のデータを取得します
            data_to_analyze = data_series.iloc[3:]
            
            if data_to_analyze.empty:
                print(f"警告: ファイル {os.path.basename(file)} の有効データがありません。")
                continue

            # 最大値 (Watt) の検出
            max_val_w = data_to_analyze.max()
            
            # dBmへの変換: 10 * log10(W) + 30
            # log(0)や負の数を避けるためのチェック
            if max_val_w > 0:
                max_val_dbm = 10 * np.log10(max_val_w) + 30
            else:
                max_val_dbm = -np.inf
            
            # SNRの計算: Signal (dBm) - Noise Floor (dBm)
            snr = max_val_dbm - noise_floor_dbm
            
            # 無限大（-inf）でない場合のみリストに追加
            if snr > -1000:  # 極端に低い値を除外する簡易チェック
                snr_values.append(snr)
            
        except Exception as e:
            print(f"エラー: {os.path.basename(file)} の処理中に問題が発生しました: {e}")

    # 平均SNRの計算と出力
    if snr_values:
        average_snr = np.mean(snr_values)
        print("-" * 30)
        print(f"計算完了")
        print(f"有効ファイル数: {len(snr_values)}")
        print(f"平均 SNR: {average_snr:.2f} dB")
        print("-" * 30)
    else:
        print("有効なSNR値を計算できませんでした。")

# --- 実行設定 ---
# フォルダパスを適切に書き換えてください
# 例: target_folder = '0.5nano' 
# 現在のディレクトリにある場合は '.' またはフォルダ名を指定
target_folder = 'ped/2.545nano/far' 

# 実行
if __name__ == "__main__":
    calculate_average_snr(target_folder)

処理対象ファイル数: 301
------------------------------
計算完了
有効ファイル数: 301
平均 SNR: -9.71 dB
------------------------------


In [17]:
#SNR=-9.71  # dB
SNR = 6
SNR = 10**(SNR/10)  # 線形値に変換
c=299792458
B = 400*1e+6  
CRLB = 3/(SNR*(np.pi**2)*(B**2))
sigma_R=c/2*np.sqrt(CRLB)
print("Range_CRLB:", sigma_R)

Range_CRLB: 0.10354794171982215


In [16]:
T_symbol = 5.575 * 1e-6
f_carrier = 28 * 1e+9
l_speed=299792458
lam = l_speed/f_carrier
P= 89
sigma_v = lam/(4*np.pi*T_symbol)*np.sqrt(6/(SNR*P**3))
print("Velocity_CRLB:", sigma_v)

Velocity_CRLB: 0.1377839246473131
